# Отчёт 4. Стохастическая и популяционная оптимизация

Имитация отжига и рой частиц на тестовых функциях, анализ гиперпараметров и применение имитации отжига к задаче N ферзей.

> **Воспроизводимость.** Для воспроизведения перезапустите ядро и выполните все ячейки по порядку.

[Описание, результаты и инструкция по запуску](../docs/04-stochastic-methods.md)


# Базовая часть


## Функции и их производные

### Функция Растригина (Rastrigin)

$$
f(x, y) = 20 + x^2 + y^2 - 10 \bigl( \cos(2\pi x) + \cos(2\pi y) \bigr)
$$

Глобальный минимум достигается в точке $(0, 0)$:

$$
f(0, 0) = 20 + 0 + 0 - 10(1 + 1) = 20 - 20 = 0
$$

#### Градиент

$$
\nabla f(x, y) = 
\begin{pmatrix}
\dfrac{\partial f}{\partial x} \\[6pt]
\dfrac{\partial f}{\partial y}
\end{pmatrix}
=
\begin{pmatrix}
2x + 20\pi \sin(2\pi x) \\[4pt]
2y + 20\pi \sin(2\pi y)
\end{pmatrix}
$$



### Функция "Ящик для яиц" (Eggcrate)

$$
f(x, y) = x^2 + y^2 + 25 \left( \sin^2 x + \sin^2 y \right)
$$

Глобальный минимум достигается в точке $(0, 0)$:

$$
f(0, 0) = 0^2 + 0^2 + 25(\sin^2 0 + \sin^2 0) = 0
$$

#### Градиент

$$
\nabla f(x, y) = 
\begin{pmatrix}
\dfrac{\partial f}{\partial x} \\[6pt]
\dfrac{\partial f}{\partial y}
\end{pmatrix}
=
\begin{pmatrix}
2x + 25 \sin 2x \\[4pt]
2y + 25 \sin 2y
\end{pmatrix}
$$


### Функция Бута (Booth)

$$
f(x, y) = (x + 2y - 7)^2 + (2x + y - 5)^2
$$

Глобальный минимум достигается в точке $(1, 3)$:

$$
f(1, 3) = (1 + 6 - 7)^2 + (2 + 3 - 5)^2 = 0^2 + 0^2 = 0
$$

#### Градиент

$$
\nabla f(x, y) = 
\begin{pmatrix}
\dfrac{\partial f}{\partial x} \\[6pt]
\dfrac{\partial f}{\partial y}
\end{pmatrix}
=
\begin{pmatrix}
10x + 8y - 34 \\[4pt]
8x + 10y - 38
\end{pmatrix}
$$


## Первая часть

## Метод Отжига

In [ ]:
def simulated_annealing(func, bounds, start_point=None, initial_temp=1000.0, cooling_rate=0.95, max_iter=500, step_size=0.5):

    if start_point is None:
        current_x = np.array([np.random.uniform(b[0], b[1]) for b in bounds])
    else:
        current_x = np.array(start_point, dtype=float)
        
    current_f = func(current_x)
    best_x, best_f = current_x.copy(), current_f
    history_x, history_f = [current_x.copy()], [current_f]
    temp = initial_temp
    
    for _ in range(max_iter):
        neighbor_x = current_x + np.random.uniform(-step_size, step_size, size=len(current_x))
        neighbor_x = np.clip(neighbor_x, [b[0] for b in bounds], [b[1] for b in bounds])
        neighbor_f = func(neighbor_x)
        delta_f = neighbor_f - current_f
        
        if delta_f < 0 or np.random.rand() < np.exp(-delta_f / temp):
            current_x, current_f = neighbor_x.copy(), neighbor_f
            if current_f < best_f:
                best_x, best_f = current_x.copy(), current_f
                
        history_x.append(current_x.copy())
        history_f.append(current_f)
        temp *= cooling_rate
        
    return best_x, best_f, np.array(history_x), np.array(history_f)

### Комментарий по отжигу
* initial_temp ($T_0$) — начальная температура системы, определяющая вероятность принятия худших решений на старте.
* cooling_rate ($\alpha$) — коэффициент экспоненциального охлаждения ($T_{new} = T \cdot \alpha$), регулирует скорость снижения температуры.
* max_iter — максимальное количество итераций алгоритма.
* step_size — максимальное случайное смещение для генерации соседней точки.
* current_x, current_f — текущие координаты и значение целевой функции.
* neighbor_x, neighbor_f — координаты и значение функции соседней сгенерированной точки.
* delta_f — изменение значения функции ($\Delta f = f_{neighbor} - f_{current}$).
* best_x, best_f — глобально лучшие найденные координаты и минимальное значение функции за всё время работы.

## Метод имитации роя

In [ ]:
def particle_swarm(func, bounds, start_point=None, n_particles=30, max_iter=100, w=0.7, c1=1.5, c2=1.5):
    dim = len(bounds)
    lower, upper = np.array([b[0] for b in bounds]), np.array([b[1] for b in bounds])
    
    if start_point is not None:
        center = np.array(start_point, dtype=float)
        spread = (upper - lower) * 0.15
        particles = np.random.uniform(center - spread, center + spread, (n_particles, dim))
    else:
        particles = np.random.uniform(lower, upper, (n_particles, dim))
        
    particles = np.clip(particles, lower, upper)
    velocities = np.random.uniform(-1, 1, (n_particles, dim))
    
    personal_best_pos = particles.copy()
    personal_best_val = np.array([func(p) for p in particles])
    
    global_best_idx = np.argmin(personal_best_val)
    global_best_pos = personal_best_pos[global_best_idx].copy()
    global_best_val = personal_best_val[global_best_idx]
    
    history_best_f = [global_best_val]
    history_mean_pos = [np.mean(particles, axis=0).copy()]
    
    for _ in range(max_iter):
        r1, r2 = np.random.rand(n_particles, dim), np.random.rand(n_particles, dim)
        
        velocities = w * velocities + c1 * r1 * (personal_best_pos - particles) + c2 * r2 * (global_best_pos - particles)
        velocities = np.clip(velocities, -(upper - lower) * 0.3, (upper - lower) * 0.3)
        particles = np.clip(particles + velocities, lower, upper)
        
        current_vals = np.array([func(p) for p in particles])
        improved = current_vals < personal_best_val
        personal_best_pos[improved] = particles[improved]
        personal_best_val[improved] = current_vals[improved]
        
        best_idx = np.argmin(personal_best_val)
        if personal_best_val[best_idx] < global_best_val:
            global_best_val = personal_best_val[best_idx]
            global_best_pos = personal_best_pos[best_idx].copy()
            
        history_best_f.append(global_best_val)
        history_mean_pos.append(np.mean(particles, axis=0).copy())
        
    return global_best_pos, global_best_val, np.array(history_best_f), np.array(history_mean_pos)

### Комментарий по имитации роя

* n_particles — размер популяции (количество частиц в рое).
* w — коэффициент инерции, определяющий влияние предыдущей скорости частицы на её новое движение.
* c1 — когнитивный коэффициент, регулирующий притяжение частицы к её личному рекорду ($pBest$).
* c2 — социальный коэффициент, регулирующий притяжение частицы к лучшему решению, найденному всем роем ($gBest$).
* particles — матрица текущих координат всех частиц.
* velocities — матрица текущих векторов скоростей всех частиц.
* personal_best_pos — массив лучших позиций, найденных каждой частицей индивидуально.
* global_best_pos — лучшая позиция, найденная любой частицей в рое на текущий момент.

## Код для запуска

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100

# ==========================================
# 1. Тестовые функции
# ==========================================
def rastrigin(x):
    return 20 + x[0]**2 + x[1]**2 - 10 * (np.cos(2 * np.pi * x[0]) + np.cos(2 * np.pi * x[1]))

def eggcrate(x):
    return x[0]**2 + x[1]**2 + 25 * (np.sin(x[0])**2 + np.sin(x[1])**2)

def booth(x):
    return (x[0] + 2 * x[1] - 7)**2 + (2 * x[0] + x[1] - 5)**2

FUNCTIONS = {
    'Rastrigin': (rastrigin, [(-5, 5), (-5, 5)]),
    'Eggcrate':  (eggcrate,  [(-5, 5), (-5, 5)]),
    'Booth':     (booth,     [(-10, 10), (-10, 10)])
}

# ==========================================
# 2. Метод имитации отжига
# ==========================================
def simulated_annealing(func, bounds, start_point=None, initial_temp=1000.0,
                        cooling_rate=0.95, max_iter=500, step_size=0.5):
    if start_point is None:
        current_x = np.array([np.random.uniform(b[0], b[1]) for b in bounds])
    else:
        current_x = np.array(start_point, dtype=float)

    current_f = func(current_x)
    best_x, best_f = current_x.copy(), current_f
    history_x, history_f = [current_x.copy()], [current_f]
    temp = initial_temp

    for _ in range(max_iter):
        neighbor_x = current_x + np.random.uniform(-step_size, step_size, size=len(current_x))
        neighbor_x = np.clip(neighbor_x, [b[0] for b in bounds], [b[1] for b in bounds])
        neighbor_f = func(neighbor_x)
        delta_f = neighbor_f - current_f
        
        if delta_f < 0 or np.random.rand() < np.exp(-delta_f / temp):
            current_x, current_f = neighbor_x.copy(), neighbor_f
            if current_f < best_f:
                best_x, best_f = current_x.copy(), current_f
                
        history_x.append(current_x.copy())
        history_f.append(current_f)
        temp *= cooling_rate

    return best_x, best_f, np.array(history_x), np.array(history_f)

# ==========================================
# 3. Параметры и стартовые точки
# ==========================================
PARAMS = {
    'Базовые': {'initial_temp': 1000.0, 'cooling_rate': 0.95, 'max_iter': 300, 'step_size': 0.5},
    'Улучшенные': {'initial_temp': 10000.0, 'cooling_rate': 0.999, 'max_iter': 5000, 'step_size': 0.5}
}

START_POINTS = {
    'Rastrigin': [(4.0, 4.0), (-3.5, 2.0), (1.5, -3.0)],
    'Eggcrate':  [(3.0, 3.0), (-2.0, 2.5), (4.0, -1.0)],
    'Booth':     [(-5.0, -5.0), (8.0, 8.0), (0.0, 0.0)]
}

# ==========================================
# 4. Запуск экспериментов
# ==========================================
rows = []
trajectories = {name: {pname: [] for pname in PARAMS} for name in FUNCTIONS}

for name, (func, bounds) in FUNCTIONS.items():
    for sp_idx, sp in enumerate(START_POINTS[name]):
        for pname, params in PARAMS.items():
            np.random.seed(42 + sp_idx)
            best_x, best_f, hist_x, hist_f = simulated_annealing(func, bounds, start_point=sp, **params)
            trajectories[name][pname].append({
                'start': sp, 'hist_x': hist_x, 'hist_f': hist_f,
                'best_x': best_x, 'best_f': best_f
            })
            rows.append({
                'Функция': name, 'Параметры': pname, 'Старт №': sp_idx + 1,
                'Старт (x, y)': f'({sp[0]}, {sp[1]})',
                'Минимум (x, y)': f'({best_x[0]:.3f}, {best_x[1]:.3f})',
                'f(min)': f'{best_f:.6f}'
            })

df_results = pd.DataFrame(rows)

# ==========================================
# 5. Графики траекторий
# ==========================================
colors_sp = ['#d62728', '#ff7f0e', '#2ca02c'] 

fig_base, axes_base = plt.subplots(1, 3, figsize=(18, 5))
for idx, (name, (func, bounds)) in enumerate(FUNCTIONS.items()):
    ax = axes_base[idx]
    x = np.linspace(bounds[0][0], bounds[0][1], 300)
    y = np.linspace(bounds[1][0], bounds[1][1], 300)
    X, Y = np.meshgrid(x, y)
    Z = np.vectorize(lambda a, b: func([a, b]))(X, Y)
    
    ax.contourf(X, Y, Z, levels=50, cmap='viridis', alpha=0.7)
    ax.contour(X, Y, Z, levels=20, colors='white', alpha=0.3, linewidths=0.5)
    
    for i, traj in enumerate(trajectories[name]['Базовые']):
        hx = traj['hist_x']
        ax.plot(hx[:, 0], hx[:, 1], '-', color=colors_sp[i], alpha=0.8, linewidth=1.5, label=f'Старт {i+1}')
        ax.plot(hx[0, 0], hx[0, 1], 'o', color='black', markersize=8, zorder=10) 
        ax.plot(traj['best_x'][0], traj['best_x'][1], 'o', color='red', markersize=8, zorder=10) 
    
    ax.set_title(f'{name}: Базовые параметры', fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Траектории метода имитации отжига: БАЗОВЫЕ параметры (T₀=1000, α=0.95)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

fig_adv, axes_adv = plt.subplots(1, 3, figsize=(18, 5))
for idx, (name, (func, bounds)) in enumerate(FUNCTIONS.items()):
    ax = axes_adv[idx]
    x = np.linspace(bounds[0][0], bounds[0][1], 300)
    y = np.linspace(bounds[1][0], bounds[1][1], 300)
    X, Y = np.meshgrid(x, y)
    Z = np.vectorize(lambda a, b: func([a, b]))(X, Y)
    
    ax.contourf(X, Y, Z, levels=50, cmap='viridis', alpha=0.7)
    ax.contour(X, Y, Z, levels=20, colors='white', alpha=0.3, linewidths=0.5)
    
    for i, traj in enumerate(trajectories[name]['Улучшенные']):
        hx = traj['hist_x']
        ax.plot(hx[:, 0], hx[:, 1], '-', color=colors_sp[i], alpha=0.8, linewidth=1.5, label=f'Старт {i+1}')
        ax.plot(hx[0, 0], hx[0, 1], 'o', color='black', markersize=8, zorder=10) # Старт
        ax.plot(traj['best_x'][0], traj['best_x'][1], 'o', color='blue', markersize=8, zorder=10) # Финиш
    
    ax.set_title(f'{name}: Улучшенные параметры', fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Траектории метода имитации отжига: УЛУЧШЕННЫЕ параметры (T₀=10000, α=0.999)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# ==========================================
# 6. Графики сходимости 
# ==========================================
fig_conv, axes_conv = plt.subplots(1, 3, figsize=(18, 5))
for idx, (name, _) in enumerate(FUNCTIONS.items()):
    ax = axes_conv[idx]

    for i, traj in enumerate(trajectories[name]['Базовые']):
        ax.plot(traj['hist_f'], '--', color=colors_sp[i], alpha=0.6, linewidth=1)

    for i, traj in enumerate(trajectories[name]['Улучшенные']):
        ax.plot(traj['hist_f'], '-', color=colors_sp[i], alpha=0.9, linewidth=1.5)
        
    ax.set_title(f'{name}: Сходимость (пунктир=баз, сплошн=улучш)', fontweight='bold')
    ax.set_yscale('log')
    ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

# ==========================================
# 7. Таблицы результатов
# ==========================================
display(df_results)

summary = []
for name in FUNCTIONS:
    for pname in PARAMS:
        vals = [t['best_f'] for t in trajectories[name][pname]]
        summary.append({
            'Функция': name, 'Параметры': pname,
            'Среднее f(min)': f'{np.mean(vals):.6f}',
            'Лучшее f(min)': f'{np.min(vals):.6f}'
        })
display(pd.DataFrame(summary))

## Выводы по базовому заданию 

### 1. Сравнение эффективности параметров
Для оценки влияния гиперпараметров были проведены запуски с двумя конфигурациями:
*   Базовые: $T_0 = 1000$, $\alpha = 0.95$, 300 итераций (быстрое охлаждение).
*   Улучшенные: $T_0 = 10000$, $\alpha = 0.999$, 5000 итераций (медленное охлаждение).

### 2. Результаты на многомодальных функциях (Rastrigin и Eggcrate)
На функциях со множеством локальных минимумов улучшенные параметры демонстрируют подавляющее преимущество:

*   Функция Eggcrate: 
    *   При базовых параметрах алгоритм застревает в локальных минимумах со средним значением $f \approx 12.7$.
    *   При улучшенных параметрах среднее значение падает до $f \approx 0.04$, а лучшее найденное решение достигает $f \approx 0.012$, что крайне близко к глобальному минимуму $f(0,0)=0$.
*   Функция Rastrigin:
    *   Улучшенные параметры позволили снизить среднюю ошибку с $4.05$ до $0.97$. 
    *   Лучший результат при улучшенных параметрах ($f \approx 0.13$) значительно точнее, чем лучший результат при базовых ($f \approx 1.01$).

Высокая начальная температура и медленное охлаждение позволяют алгоритму дольше успешно покидать глубокие локальные ловушки, в которые быстро попадает алгоритм с быстрым охлаждением.

### 3. Результаты на выпуклой функции (Booth)
На функции Бута наблюдается обратная тенденция:
*   Базовые параметры находят решение с высокой точностью ($f \approx 10^{-6}$).
*   Улучшенные параметры показывают худшую точность ($f \approx 0.13$ в среднем).

Функция Бута не имеет локальных минимумов. Быстрое охлаждение базового метода позволяет алгоритму быстро скатиться в глобальный минимум. Улучшенный метод из-за высокой температуры продолжает совершать большие случайные шаги даже вблизи минимума, не успевая точно стабилизироваться до самого конца.

### 4. Итог
Метод имитации отжига требует тонкой настройки под тип задачи:
1.  Для поиска глобального минимума среди многих локальных критически важны высокая $T_0$ и медленное охлаждение. Без этого алгоритм застревает.
2.  Для выпуклых задач излишняя теплота системы мешает.

# Extension-1

## Код для запуска

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ==========================================
# 1. Тестовые функции
# ==========================================
def rastrigin(x):
    return 20 + x[0]**2 + x[1]**2 - 10 * (np.cos(2 * np.pi * x[0]) + np.cos(2 * np.pi * x[1]))

def eggcrate(x):
    return x[0]**2 + x[1]**2 + 25 * (np.sin(x[0])**2 + np.sin(x[1])**2)

def booth(x):
    return (x[0] + 2 * x[1] - 7)**2 + (2 * x[0] + x[1] - 5)**2

FUNCTIONS = {
    'Rastrigin': (rastrigin, [(-5, 5), (-5, 5)]),
    'Eggcrate':  (eggcrate,  [(-5, 5), (-5, 5)]),
    'Booth':     (booth,     [(-10, 10), (-10, 10)])
}

# ==========================================
# 2. Методы
# ==========================================
def simulated_annealing(func, bounds, start_point=None, initial_temp=1000.0, cooling_rate=0.95, max_iter=500, step_size=0.5):
    if start_point is None:
        current_x = np.array([np.random.uniform(b[0], b[1]) for b in bounds])
    else:
        current_x = np.array(start_point, dtype=float)
    current_f = func(current_x)
    best_x, best_f = current_x.copy(), current_f
    history_x, history_f = [current_x.copy()], [current_f]
    temp = initial_temp
    for _ in range(max_iter):
        neighbor_x = current_x + np.random.uniform(-step_size, step_size, size=len(current_x))
        neighbor_x = np.clip(neighbor_x, [b[0] for b in bounds], [b[1] for b in bounds])
        neighbor_f = func(neighbor_x)
        delta_f = neighbor_f - current_f
        if delta_f < 0 or np.random.rand() < np.exp(-delta_f / temp):
            current_x, current_f = neighbor_x.copy(), neighbor_f
            if current_f < best_f: best_x, best_f = current_x.copy(), current_f
        history_x.append(current_x.copy())
        history_f.append(current_f)
        temp *= cooling_rate
    return best_x, best_f, np.array(history_x), np.array(history_f)

def particle_swarm(func, bounds, start_point=None, n_particles=30, max_iter=100, w=0.7, c1=1.5, c2=1.5):
    dim = len(bounds)
    lower, upper = np.array([b[0] for b in bounds]), np.array([b[1] for b in bounds])
    if start_point is not None:
        center = np.array(start_point, dtype=float)
        spread = (upper - lower) * 0.15
        particles = np.random.uniform(center - spread, center + spread, (n_particles, dim))
    else:
        particles = np.random.uniform(lower, upper, (n_particles, dim))
    particles = np.clip(particles, lower, upper)
    velocities = np.random.uniform(-1, 1, (n_particles, dim))
    personal_best_pos, personal_best_val = particles.copy(), np.array([func(p) for p in particles])
    global_best_idx = np.argmin(personal_best_val)
    global_best_pos, global_best_val = personal_best_pos[global_best_idx].copy(), personal_best_val[global_best_idx]
    history_best_f, history_mean_pos = [global_best_val], [np.mean(particles, axis=0).copy()]
    
    for _ in range(max_iter):
        r1, r2 = np.random.rand(n_particles, dim), np.random.rand(n_particles, dim)
        velocities = w * velocities + c1 * r1 * (personal_best_pos - particles) + c2 * r2 * (global_best_pos - particles)
        velocities = np.clip(velocities, -(upper - lower) * 0.3, (upper - lower) * 0.3)
        particles = np.clip(particles + velocities, lower, upper)
        current_vals = np.array([func(p) for p in particles])
        improved = current_vals < personal_best_val
        personal_best_pos[improved], personal_best_val[improved] = particles[improved], current_vals[improved]
        best_idx = np.argmin(personal_best_val)
        if personal_best_val[best_idx] < global_best_val:
            global_best_val, global_best_pos = personal_best_val[best_idx], personal_best_pos[best_idx].copy()
        history_best_f.append(global_best_val)
        history_mean_pos.append(np.mean(particles, axis=0).copy())
    return global_best_pos, global_best_val, np.array(history_best_f), np.array(history_mean_pos)

# ==========================================
# 3. Запуск
# ==========================================
START_POINTS = {
    'Rastrigin': [(4.0, 4.0), (-3.5, 2.0), (1.5, -3.0)],
    'Eggcrate':  [(3.0, 3.0), (-2.0, 2.5), (4.0, -1.0)],
    'Booth':     [(-5.0, -5.0), (8.0, 8.0), (0.0, 0.0)]
}
rows, trajectories = [], {name: {'SA': [], 'PSO': []} for name in FUNCTIONS}

for name, (func, bounds) in FUNCTIONS.items():
    for sp_idx, sp in enumerate(START_POINTS[name]):
        np.random.seed(42 + sp_idx)
        best_x_sa, best_f_sa, hist_x_sa, hist_f_sa = simulated_annealing(func, bounds, start_point=sp, initial_temp=1000.0, cooling_rate=0.95, max_iter=300, step_size=0.5)
        trajectories[name]['SA'].append({'start': sp, 'hist_x': hist_x_sa, 'hist_f': hist_f_sa, 'best_x': best_x_sa, 'best_f': best_f_sa})
        rows.append({'Функция': name, 'Метод': 'SA', 'Старт №': sp_idx + 1, 'f(min)': f'{best_f_sa:.4f}'})
        
        np.random.seed(42 + sp_idx)
        best_x_pso, best_f_pso, hist_f_pso, hist_mean_pso = particle_swarm(func, bounds, start_point=sp, n_particles=30, max_iter=100, w=0.7, c1=1.5, c2=1.5)
        trajectories[name]['PSO'].append({'start': sp, 'hist_mean': hist_mean_pso, 'hist_f': hist_f_pso, 'best_x': best_x_pso, 'best_f': best_f_pso})
        rows.append({'Функция': name, 'Метод': 'PSO', 'Старт №': sp_idx + 1, 'f(min)': f'{best_f_pso:.4f}'})

df_results = pd.DataFrame(rows)

# ==========================================
# 4. Графики
# ==========================================
fig, axes = plt.subplots(3, 1, figsize=(14, 22))
colors_sa, colors_pso = ['#d62728', '#ff7f0e', '#e377c2'], ['#1f77b4', '#17becf', '#9467bd']

for idx, (name, (func, bounds)) in enumerate(FUNCTIONS.items()):
    ax = axes[idx]
    x = np.linspace(bounds[0][0], bounds[0][1], 400)
    y = np.linspace(bounds[1][0], bounds[1][1], 400)
    X, Y = np.meshgrid(x, y)
    Z = np.vectorize(lambda a, b: func([a, b]))(X, Y)
    cf = ax.contourf(X, Y, Z, levels=60, cmap='viridis', alpha=0.75)
    plt.colorbar(cf, ax=ax, fraction=0.046, pad=0.04)
    ax.contour(X, Y, Z, levels=25, colors='white', alpha=0.35, linewidths=0.5)
    
    for i, traj in enumerate(trajectories[name]['SA']):
        hx = traj['hist_x']
        ax.plot(hx[:, 0], hx[:, 1], '-', color=colors_sa[i], alpha=0.75, linewidth=1.5, label=f'SA старт {i+1}')
        ax.plot(hx[0, 0], hx[0, 1], 'o', color='black', markersize=10, zorder=10)
        ax.plot(traj['best_x'][0], traj['best_x'][1], 'o', color='red', markersize=10, zorder=10)
    
    for i, traj in enumerate(trajectories[name]['PSO']):
        hm = traj['hist_mean']
        ax.plot(hm[:, 0], hm[:, 1], '--', color=colors_pso[i], alpha=0.75, linewidth=1.5, label=f'PSO старт {i+1}')
        ax.plot(hm[0, 0], hm[0, 1], 'o', color='black', markersize=10, zorder=10)
        ax.plot(traj['best_x'][0], traj['best_x'][1], 'o', color='red', markersize=10, zorder=10)
    
    ax.set_title(f'{name}: сравнение SA и PSO (3 запуска каждый)', fontsize=14, fontweight='bold')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(loc='upper right', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

# ==========================================
# 5. Исследование гиперпараметров
# ==========================================
swarm_sizes = [10, 20, 30, 50]
c_values = [0.5, 1.5, 2.5]
results_all = []

for name, (func, bounds) in FUNCTIONS.items():
    for c in c_values:
        for n in swarm_sizes:
            best_vals = [particle_swarm(func, bounds, n_particles=n, max_iter=100, w=0.7, c1=c, c2=c)[1] for _ in range(5)]
            results_all.append({
                'Функция': name,
                'Размер роя': n,
                'c1=c2': c,
                'Средняя f(x)': np.mean(best_vals)
            })

df_pso_all = pd.DataFrame(results_all)
df_pso_all['Функция'] = pd.Categorical(df_pso_all['Функция'], categories=list(FUNCTIONS.keys()), ordered=True)
df_pso_all = df_pso_all.sort_values(by=['Функция', 'c1=c2', 'Размер роя']).reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
EPS = 1e-20

for idx, (name, (func, bounds)) in enumerate(FUNCTIONS.items()):
    ax = axes[idx]
    sub = df_pso_all[df_pso_all['Функция'] == name]
    
    for c in c_values:
        sub_c = sub[sub['c1=c2'] == c].sort_values('Размер роя')
        y_vals = [max(v, EPS) for v in sub_c['Средняя f(x)'].values]
        ax.plot(sub_c['Размер роя'], y_vals, marker='o', label=f'c1=c2={c}', linewidth=2)
    
    ax.set_title(f'Гиперпараметры PSO: {name}')
    ax.set_xlabel('Размер роя')
    ax.set_ylabel('Средняя f(x) (log-шкала)')
    ax.set_xticks(swarm_sizes)
    ax.set_yscale('log')
    ax.legend(fontsize=8)
    ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

# ==========================================
# 6. Таблицы результатов
# ==========================================
display(df_results)
display(df_pso_all)

## Вывод по 2 стохастическим методам

### 1. Сравнение друг с другом

Анализ результатов показывает принципиально разное поведение методов имитации отжига (SA) и роя частиц (PSO):

*   **Точность и сходимость:**
    *   PSO демонстрирует значительно более высокую точность. На функциях Rastrigin и Booth метод находит решения, близкие к машинному нулю ($f \approx 10^{-15}$), что свидетельствует о надежном нахождении глобального минимума.
    *   SA в базовой конфигурации ($T_0=1000, \alpha=0.95$) застревает в локальных минимумах. На функции Rastrigin лучшие найденные значения $f \in [1.01; 8.77]$, что далеко от глобального нуля. На Eggcrate SA также не смог преодолеть локальные барьеры ($f \approx 9.5 - 19.0$).

*   Влияние гиперпараметров (для PSO):
    *   Исследование показало сильную зависимость качества решения от коэффициентов $c_1, c_2$.
    *   Значение **$c_1=c_2=0.5$** оказалось наиболее эффективным для всех трех функций, обеспечивая точность порядка $10^{-15}$. Низкие коэффициенты способствуют большему разнообразию в рое (exploration), предотвращая преждевременную сходимость.
    *   Увеличение коэффициентов до 1.5 и 2.5 приводит к резкому падению точности на сложных функциях (Rastrigin), так как частицы начинают слишком агрессивно стремиться к текущим лидерам, игнорируя исследование пространства.

*   **Стабильность:**
    *   PSO показал высокую стабильность: из разных стартовых точек он приходил к одним и тем же отличным результатам.
    *   SA сильно зависел от начальной точки: разные старты приводили к попаданию в разные локальные ямы с существенно отличающимися значениями функции.

### 2. Сравнение с другими методами

При сопоставлении с детерминированными алгоритмами (Градиентный спуск, BFGS, Нелдер-Мид) выявляются следующие закономерности:

*   На выпуклых функциях (Booth):
    *   Детерминированные методы (особенно BFGS) находят решение быстрее и точнее любых стохастических.
    *   Однако PSO также справляется с этой задачей идеально ($f \approx 0$), в то время как базовый SA дает небольшую погрешность ($f \approx 0.007$).

*   На мнультимодальных функциях (Rastrigin, Eggcrate):
    *   Детерминированные методы использующие градиент (GD, BFGS) если скатываются в локальный минимум,то не имеют механизмов выхода из него. Их результат критически зависит от стартовой точки и далёк от глобального оптимума, при неудачном выборе точки.
    *   Нелдер-Мид, хотя и не использует градиент, также является методом локального поиска и часто застревает, уступая PSO.
    *   PSO самый крутой, находит глобальный минимум там, где методы прошлых лабораторных улетают в локальный минимум.
    *   SA не хуже детерминированные методы по точности на сложных ландшафтах при правильной настройке параметров, однако очень дорогой по вычислениям.

Итог: Для задач с большим количеством локальных минимумов метод роя частиц (PSO) является наиболее предпочтительным среди рассмотренных благодаря исследования всего окружающего пространства и механизма выхода из локальной ямы. Классические градиентные методы будут лучше по скорости только для простых выпуклых задач.

# Extension-2


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. Реализация задачи об N ферзях
# ==========================================
def count_conflicts(perm):
    n = len(perm)
    idx = np.arange(n)
    main_diag = idx - perm + (n - 1)
    anti_diag = idx + perm
    counts_main = np.bincount(main_diag, minlength=2 * n - 1)
    counts_anti = np.bincount(anti_diag, minlength=2 * n - 1)
    conflicts = (np.sum(counts_main * (counts_main - 1) // 2) + np.sum(counts_anti * (counts_anti - 1) // 2))
    return int(conflicts)

def n_queens_sa(n=100, max_iter=100000, initial_temp=10.0, cooling_rate=0.99995):
    np.random.seed(42)
    perm = np.random.permutation(n)
    current_conflicts = count_conflicts(perm)
    best_perm, best_conflicts = perm.copy(), current_conflicts
    history, temp, found_at = [current_conflicts], initial_temp, None

    for i in range(max_iter):
        if best_conflicts == 0:
            found_at = i
            break
        new_perm = perm.copy()
        i1, i2 = np.random.choice(n, 2, replace=False)
        new_perm[i1], new_perm[i2] = new_perm[i2], new_perm[i1]
        new_conflicts = count_conflicts(new_perm)
        delta = new_conflicts - current_conflicts
        
        if delta < 0 or np.random.rand() < np.exp(-delta / temp):
            perm, current_conflicts = new_perm, new_conflicts
            if current_conflicts < best_conflicts:
                best_perm, best_conflicts = perm.copy(), current_conflicts
        history.append(current_conflicts)
        temp *= cooling_rate
    return best_perm, best_conflicts, history, found_at

# ==========================================
# 2. Запуск метода
# ==========================================
N = 100
best_perm, best_conflicts, history, found_at = n_queens_sa(n=N, max_iter=100000, initial_temp=10.0, cooling_rate=0.99995)

if found_at is not None:
    print(f"Идеальное решение найдено на итерации {found_at}!\n")
else:
    print(f"Найден почти оптимальный вариант: конфликтов = {best_conflicts}\n")

# ==========================================
# 3. Визуализация решения
# ==========================================
fig = plt.figure(figsize=(16, 7))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.2])

ax1 = fig.add_subplot(gs[0, 0])
board = np.zeros((N, N))
for i in range(N):
    board[i, best_perm[i]] = 1
for i in range(N):
    for j in range(N):
        if (i + j) % 2 == 0:
            board[i, j] = 0.15 if board[i, j] == 0 else 1

ax1.imshow(board, cmap='gray', origin='lower', aspect='equal')

if best_conflicts > 0:
    conflict_cells = set()
    for i in range(N):
        for j in range(i + 1, N):
            if abs(i - j) == abs(best_perm[i] - best_perm[j]):
                conflict_cells.add((i, best_perm[i]))
                conflict_cells.add((j, best_perm[j]))
    for (r, c) in conflict_cells:
        ax1.plot(c, r, 'o', color='red', markersize=6, alpha=0.8)

ax1.set_title(f'Решение задачи {N} ферзей\nконфликтов: {best_conflicts}', fontsize=14, fontweight='bold')
ax1.set_xticks([]); ax1.set_yticks([])

ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(history, color='#1f77b4', linewidth=1.2)
ax2.set_title('Сходимость метода имитации отжига', fontsize=14, fontweight='bold')
ax2.set_xlabel('итерация', fontsize=12)
ax2.set_ylabel('количество конфликтов', fontsize=12)
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)
ax2.axhline(y=0, color='green', linestyle='--', alpha=0.5, label='0')
ax2.legend()

plt.tight_layout()
plt.show()


## Вывод
Была выбрана задача о размещении на доске N * N максимальное (N, при размерах доски > 4) ферзей. Видно, что методу требуется довольно много итераций, чтобы найти идеальное решение, однако он его находит. Можно ради интереса оценить асимптотику решения: неоптимально было бы перебирать пары ферзей, ушло бы O(N^2) времени, можно воспользоваться np.bincount(замечательная функция из numpy), которая считает за O(N), а именно нумерует количество диагоналей и каждый ферзь стоит на двух. поэтому проверка за O(N + 2N - 1) = O(N) 